In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

Mounted at /content/drive


In [ ]:
enron = pd.read_csv(f'{BASE_PATH}/enron_emails_cleaned_temporal.csv')
nazario = pd.read_csv(f'{BASE_PATH}/nazario5_cleaned_temporal.csv')
phishtank = pd.read_csv(f'{BASE_PATH}/PhishTank_2026_processed.csv')
email_text = pd.read_csv(f'{BASE_PATH}/email_text_cleaned.csv')

print("Enron:", enron.shape)
print("Nazario:", nazario.shape)
print("PhishTank:", phishtank.shape)
print("email_text:", email_text.shape)

Enron: (250610, 33)
Nazario: (3012, 31)
PhishTank: (64280, 11)
email_text: (53661, 12)


In [ ]:
# Enron: sender = 'from', date = 'utc_datetime'
enron['utc_datetime'] = pd.to_datetime(enron['utc_datetime'], errors='coerce')
enron['sender_norm'] = enron['from'].astype(str).str.strip().str.lower()

# Nazario: sender = 'sender_address' (already parsed out), date = 'utc_datetime'
nazario['utc_datetime'] = pd.to_datetime(nazario['utc_datetime'], errors='coerce')
nazario['sender_norm'] = nazario['sender_address'].astype(str).str.strip().str.lower()

# drop rows where datetime failed to parse -- can't compute temporal features without it
enron_valid = enron[enron['utc_datetime'].notna()].copy()
nazario_valid = nazario[nazario['utc_datetime'].notna()].copy()

print(f"Enron: {len(enron_valid)}/{len(enron)} rows have valid datetime")
print(f"Nazario: {len(nazario_valid)}/{len(nazario)} rows have valid datetime")

Enron: 250249/250610 rows have valid datetime
Nazario: 2980/3012 rows have valid datetime


In [ ]:
def compute_sender_features(df, sender_col='sender_norm', date_col='utc_datetime'):
    """
    Computes per-sender aggregate features and merges them back onto every row.
    All features here are REAL -- derived directly from existing date/sender data,
    no synthetic generation involved.
    """
    df = df.sort_values(date_col).copy()

    sender_stats = df.groupby(sender_col)[date_col].agg(
        sender_email_count='count',
        sender_first_seen='min',
        sender_last_seen='max'
    ).reset_index()

    sender_stats['sender_active_days'] = (
        sender_stats['sender_last_seen'] - sender_stats['sender_first_seen']
    ).dt.total_seconds() / 86400
    # avoid divide-by-zero for senders seen on only one day
    sender_stats['sender_active_days'] = sender_stats['sender_active_days'].clip(lower=1)

    sender_stats['sender_avg_daily_volume'] = (
        sender_stats['sender_email_count'] / sender_stats['sender_active_days']
    )

    df = df.merge(sender_stats, on=sender_col, how='left')
    return df

enron_feat = compute_sender_features(enron_valid)
nazario_feat = compute_sender_features(nazario_valid)

print(enron_feat[['sender_norm','sender_email_count','sender_active_days','sender_avg_daily_volume']].head())

             sender_norm  sender_email_count  sender_active_days  sender_avg_daily_volume
0    laurinh@prodigy.net                   4         1846.751076                 0.002166
1  steven.kean@enron.com                1376         1599.727083                 0.860147
2  steven.kean@enron.com                1376         1599.727083                 0.860147
3  steven.kean@enron.com                1376         1599.727083                 0.860147
4  steven.kean@enron.com                1376         1599.727083                 0.860147


In [ ]:
def compute_interarrival_features(df, sender_col='sender_norm', date_col='utc_datetime'):
    df = df.sort_values([sender_col, date_col]).copy()
    df['prev_email_time'] = df.groupby(sender_col)[date_col].shift(1)
    df['interarrival_hours'] = (
        (df[date_col] - df['prev_email_time']).dt.total_seconds() / 3600
    )

    # per-sender baseline: median inter-arrival gap (robust to outliers vs mean)
    sender_median_gap = df.groupby(sender_col)['interarrival_hours'].transform('median')
    df['interarrival_ratio'] = df['interarrival_hours'] / sender_median_gap.replace(0, np.nan)

    # flag as anomalous burst if this gap is under 10% of the sender's usual rhythm
    # (needs at least a few emails from that sender to be meaningful)
    df['is_burst_anomaly'] = (
        (df['interarrival_ratio'] < 0.1) & (df['sender_email_count'] >= 5)
    ).fillna(False)

    return df

enron_feat = compute_interarrival_features(enron_feat)
nazario_feat = compute_interarrival_features(nazario_feat)

print(enron_feat[['sender_norm','interarrival_hours','interarrival_ratio','is_burst_anomaly']].head(10))

                           sender_norm  interarrival_hours  interarrival_ratio  is_burst_anomaly
181783   'todd'.delahoussaye@enron.com                 NaN                 NaN             False
203962   'todd'.delahoussaye@enron.com          650.188333            1.000000             False
205949   'todd'.delahoussaye@enron.com           24.664444            0.037934             False
238703   'todd'.delahoussaye@enron.com         1727.610000            2.657092             False
179502  --migrated--bmishkin@ercot.com                 NaN                 NaN             False
188667     --migrated--dodle@ercot.com                 NaN                 NaN             False
127102              -nikole@excite.com                 NaN                 NaN             False
135563              -nikole@excite.com          469.611944            4.256891             False
136402              -nikole@excite.com          116.632222            1.057236             False
139286              -nikole@ex

In [ ]:
URL_REGEX = re.compile(r'https?://[^\s<>"\')\]]+', re.IGNORECASE)

def extract_url_count(text):
    if pd.isna(text):
        return 0
    return len(URL_REGEX.findall(str(text)))

enron_feat['url_count'] = enron_feat['body'].apply(extract_url_count)

print(enron_feat['url_count'].describe())
print("Nazario url_count already present:")
print(nazario_feat['url_count'].describe())

count    250249.000000
mean          0.933486
std           6.614507
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         539.000000
Name: url_count, dtype: float64
Nazario url_count already present:
count    2980.000000
mean        1.585570
std         9.793559
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max       467.000000
Name: url_count, dtype: float64


In [ ]:
enron_unified = pd.DataFrame({
    'email_id': enron_feat['message_id'],
    'source_dataset': 'enron',
    'label': 0,  # Enron = benign baseline
    'sender': enron_feat['sender_norm'],
    'sender_domain': enron_feat['sender_norm'].str.extract(r'@([\w\.-]+)')[0],
    'datetime': enron_feat['utc_datetime'],
    'day_of_week': enron_feat['day_of_week'],
    'hour_of_day_utc': enron_feat['hour_of_day_utc'],
    'is_weekend': enron_feat['is_weekend'],
    'sender_email_count': enron_feat['sender_email_count'],
    'sender_active_days': enron_feat['sender_active_days'],
    'sender_avg_daily_volume': enron_feat['sender_avg_daily_volume'],
    'interarrival_hours': enron_feat['interarrival_hours'],
    'interarrival_ratio': enron_feat['interarrival_ratio'],
    'is_burst_anomaly': enron_feat['is_burst_anomaly'],
    'url_count': enron_feat['url_count'],
    'body_text': enron_feat['body'],
})

nazario_unified = pd.DataFrame({
    'email_id': nazario_feat['sender_norm'] + '_' + nazario_feat['utc_datetime'].astype(str),
    'source_dataset': 'nazario',
    'label': 1,  # Nazario = phishing
    'sender': nazario_feat['sender_norm'],
    'sender_domain': nazario_feat['sender_domain'],
    'datetime': nazario_feat['utc_datetime'],
    'day_of_week': nazario_feat['day_of_week'],
    'hour_of_day_utc': nazario_feat['hour_of_day_utc'],
    'is_weekend': nazario_feat['is_weekend'],
    'sender_email_count': nazario_feat['sender_email_count'],
    'sender_active_days': nazario_feat['sender_active_days'],
    'sender_avg_daily_volume': nazario_feat['sender_avg_daily_volume'],
    'interarrival_hours': nazario_feat['interarrival_hours'],
    'interarrival_ratio': nazario_feat['interarrival_ratio'],
    'is_burst_anomaly': nazario_feat['is_burst_anomaly'],
    'url_count': nazario_feat['url_count'],
    'body_text': nazario_feat['body'],
})

real_features = pd.concat([enron_unified, nazario_unified], ignore_index=True)
print(real_features.shape)
real_features.head()

(253229, 17)


,email_id,source_dataset,label,sender,sender_domain,datetime,day_of_week,hour_of_day_utc,is_weekend,sender_email_count,sender_active_days,sender_avg_daily_volume,interarrival_hours,interarrival_ratio,is_burst_anomaly,url_count,body_text
0,<26593136.1075858914131.JavaMail.evans@thyme>,enron,0,'todd'.delahoussaye@enron.com,enron.com,2001-10-24 13:50:26+00:00,2.0,13.0,False,4,100.102616,0.039959,NaN,NaN,False,0,"The below is a list of Monday, October 22 deal..."
1,<18068955.1075862169423.JavaMail.evans@thyme>,enron,0,'todd'.delahoussaye@enron.com,enron.com,2001-11-20 16:01:44+00:00,1.0,16.0,False,4,100.102616,0.039959,650.188333,1.000000,False,0,"The below is a list of Friday, November 16th d..."
2,<19670300.1075862170181.JavaMail.evans@thyme>,enron,0,'todd'.delahoussaye@enron.com,enron.com,2001-11-21 16:41:36+00:00,2.0,16.0,False,4,100.102616,0.039959,24.664444,0.037934,False,0,"The below is a list of Monday, November 19th d..."
3,<23855107.1075861104877.JavaMail.evans@thyme>,enron,0,'todd'.delahoussaye@enron.com,enron.com,2002-02-01 16:18:12+00:00,4.0,16.0,False,4,100.102616,0.039959,1727.610000,2.657092,False,0,"-----Original Message-----\nFrom: \tMulvany, P..."
4,<32216748.1075858648018.JavaMail.evans@thyme>,enron,0,--migrated--bmishkin@ercot.com,ercot.com,2001-10-22 22:10:32+00:00,0.0,22.0,False,1,1.000000,1.000000,NaN,NaN,False,0,The Portal will be down from 6-6:30pm tomorrow...


In [ ]:
print("Label distribution:\n", real_features['label'].value_counts())
print("\nMissing values per column:\n", real_features.isna().sum())
print("\nUnique sender domains:", real_features['sender_domain'].nunique())
print("\nBurst anomaly rate by label:\n", real_features.groupby('label')['is_burst_anomaly'].mean())

Label distribution:
 label
0    250249
1      2980
Name: count, dtype: int64

Missing values per column:
 email_id                       0
source_dataset                 0
label                          0
sender                         0
sender_domain                 43
datetime                       0
day_of_week                    0
hour_of_day_utc                0
is_weekend                     0
sender_email_count             0
sender_active_days             0
sender_avg_daily_volume        0
interarrival_hours         22364
interarrival_ratio         24910
is_burst_anomaly               0
url_count                      0
body_text                      0
dtype: int64

Unique sender domains: 6758

Burst anomaly rate by label:
 label
0    0.165955
1    0.021141
Name: is_burst_anomaly, dtype: float64


In [ ]:
output_path = f'{BASE_PATH}/real_features_step1.csv'
real_features.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /content/drive/My Drive/phishing project datasets/processed/real_features_step1.csv


**--------------------------------------------------------------------------------------------------**


In [ ]:
# One row per unique sender_domain, carrying the dominant label for that domain
# (a domain used in both classes is rare/noise; we take the majority label)
domain_table = (
    real_features
    .dropna(subset=['sender_domain'])
    .groupby('sender_domain')
    .agg(label=('label', lambda x: x.mode()[0]), n_emails=('label', 'count'))
    .reset_index()
)

domain_table['tld'] = domain_table['sender_domain'].str.split('.').str[-1].str.lower()

print(domain_table.shape)
print(domain_table['label'].value_counts())
domain_table.head()

(6758, 4)
label
0    5272
1    1486
Name: count, dtype: int64


,sender_domain,label,n_emails,tld
0,001mail.domain.com,1,2,com
1,03583.com,0,1,com
2,1.americanexpress.com,0,54,com
3,1.maildb.egreetings.com,0,6,com
4,10kwizard.com,0,2,com


In [ ]:
np.random.seed(42)  # reproducibility

# --- CONFIG: sourced parameters, kept explicit and separate from logic for traceability ---
DOMAIN_AGE_PARAMS = {
    'phishing': {
        # Interisle/Cybercrime Info Center, Dec 2025 phishing-domain creation-date analysis
        'p_under_30d': 0.23,
        'p_under_1y_cumulative': 0.81,
        # remaining 0.19 = aged >1yr, log-distributed per their finding that
        # older-year registrations "weren't negligible"
    },
    'benign': {
        # NOT a cited empirical stat -- documented modeling assumption:
        # established corporate domains (e.g. enron.com) are typically
        # multi-year registrations. Flag this explicitly in the thesis.
        'lognorm_mean_days': np.log(365 * 6),   # centered ~6 years
        'lognorm_sigma': 0.7
    }
}

def sample_domain_age_days(label, n):
    ages = np.empty(n)
    if label == 1:  # phishing
        p = DOMAIN_AGE_PARAMS['phishing']
        u = np.random.rand(n)
        # bucket 1: < 30 days
        mask_fresh = u < p['p_under_30d']
        # bucket 2: 30 days -- 1 year
        mask_mid = (u >= p['p_under_30d']) & (u < p['p_under_1y_cumulative'])
        # bucket 3: aged, >1 year, log-uniform out to 15 years
        mask_aged = u >= p['p_under_1y_cumulative']

        ages[mask_fresh] = np.random.uniform(1, 30, mask_fresh.sum())
        ages[mask_mid] = np.random.uniform(30, 365, mask_mid.sum())
        ages[mask_aged] = np.exp(np.random.uniform(np.log(365), np.log(365*15), mask_aged.sum()))
    else:  # benign
        params = DOMAIN_AGE_PARAMS['benign']
        ages = np.exp(np.random.normal(params['lognorm_mean_days'], params['lognorm_sigma'], n))
        ages = np.clip(ages, 30, 365*30)  # cap at 30 years, floor at 30 days

    return ages

domain_table['domain_age_days'] = domain_table.apply(
    lambda row: sample_domain_age_days(row['label'], 1)[0], axis=1
)

print(domain_table.groupby('label')['domain_age_days'].describe())

        count         mean          std         min          25%          50%          75%          max
label                                                                                                  
0      5272.0  2747.347848  1991.237225  229.739343  1381.741885  2191.102755  3502.774488  10950.00000
1      1486.0   428.186577   832.606727    1.310744    37.143727   174.851218   315.219342   5458.78451


In [ ]:
# Real phishing scores from Interisle Phishing Landscape 2025 (May 2024-Apr 2025 data)
# Score = phishing domains per capita of registrations in that TLD. .com = 30 (baseline reference).
TLD_PHISHING_SCORE = {
    'com': 30, 'org': 45, 'net': 60, 'edu': 15, 'gov': 10,   # low-abuse, established TLDs
    'info': 850, 'xyz': 1200, 'top': 1500, 'club': 900,
    'icu': 2200, 'cfd': 3000, 'bond': 1759, 'xin': 10810,     # high-abuse, cheap TLDs
}
DEFAULT_TLD_SCORE = 400  # fallback for TLDs not in the table -- moderate/unknown risk

domain_table['tld_phishing_score'] = domain_table['tld'].map(TLD_PHISHING_SCORE).fillna(DEFAULT_TLD_SCORE)

# normalize to 0-1 risk weight via log scale (raw scores span 10 to 10,810)
domain_table['tld_risk_weight'] = (
    np.log1p(domain_table['tld_phishing_score']) / np.log1p(domain_table['tld_phishing_score'].max())
)

print(domain_table[['tld', 'tld_phishing_score', 'tld_risk_weight']].drop_duplicates().sort_values('tld_risk_weight'))

            tld  tld_phishing_score  tld_risk_weight
209         gov                10.0         0.327855
116         edu                15.0         0.379086
0           com                30.0         0.469516
62          org                45.0         0.523476
7           net                60.0         0.562064
...         ...                 ...              ...
6672  xlcapital               400.0         0.819532
6736         zm               400.0         0.819532
115        info               850.0         0.922411
4398        xyz              1200.0         0.969513
1264        top              1500.0         1.000000

[145 rows x 3 columns]


In [ ]:
def compute_reputation_score(age_days, tld_risk_weight, noise_std=8):
    # age component: older = better reputation, saturates after ~2 years (730 days)
    age_component = 100 * np.minimum(age_days / 730, 1.0)

    # TLD component: higher tld_risk_weight subtracts from reputation
    tld_penalty = 40 * tld_risk_weight

    raw_score = age_component - tld_penalty
    noisy_score = raw_score + np.random.normal(0, noise_std, len(raw_score))
    return np.clip(noisy_score, 0, 100)

domain_table['domain_reputation_score'] = compute_reputation_score(
    domain_table['domain_age_days'].values,
    domain_table['tld_risk_weight'].values
)

print(domain_table.groupby('label')['domain_reputation_score'].describe())

        count       mean        std  min        25%        50%        75%    max
label                                                                           
0      5272.0  78.353135  11.316977  0.0  73.076165  79.498118  85.545538  100.0
1      1486.0  16.896509  26.589400  0.0   0.000000   1.155027  21.837576  100.0


In [ ]:
# Real base rates: DMARC.org / EasyDMARC 2026, Validity
AUTH_PARAMS = {
    'phishing': {'p_has_dmarc': 0.18, 'p_enforced_given_record': 0.08},
    'benign':   {'p_has_dmarc': 0.90, 'p_enforced_given_record': 0.44},  # corporate-tier, not global avg
}

def sample_auth_flags(label, n):
    p = AUTH_PARAMS['phishing'] if label == 1 else AUTH_PARAMS['benign']
    has_dmarc = np.random.rand(n) < p['p_has_dmarc']
    is_enforced = has_dmarc & (np.random.rand(n) < p['p_enforced_given_record'])

    # SPF/DKIM correlated with DMARC status, not independent flips:
    # if DMARC exists, SPF/DKIM alignment is likely (~85% of the time);
    # if no DMARC, SPF/DKIM presence is spottier (~30% baseline, still possible independently)
    spf_pass = np.where(has_dmarc, np.random.rand(n) < 0.85, np.random.rand(n) < 0.30)
    dkim_pass = np.where(has_dmarc, np.random.rand(n) < 0.85, np.random.rand(n) < 0.30)

    return has_dmarc, is_enforced, spf_pass, dkim_pass

has_dmarc, is_enforced, spf_pass, dkim_pass = [], [], [], []
for _, row in domain_table.iterrows():
    h, e, s, d = sample_auth_flags(row['label'], 1)
    has_dmarc.append(h[0]); is_enforced.append(e[0]); spf_pass.append(s[0]); dkim_pass.append(d[0])

domain_table['has_dmarc_record'] = has_dmarc
domain_table['dmarc_enforced'] = is_enforced
domain_table['spf_pass'] = spf_pass
domain_table['dkim_pass'] = dkim_pass

print(domain_table.groupby('label')[['has_dmarc_record','dmarc_enforced','spf_pass','dkim_pass']].mean())

       has_dmarc_record  dmarc_enforced  spf_pass  dkim_pass
label                                                       
0              0.901555        0.394917  0.801593   0.790212
1              0.182369        0.012113  0.390310   0.411171


In [ ]:
synthetic_domain_cols = [
    'sender_domain', 'domain_age_days', 'tld_phishing_score', 'tld_risk_weight',
    'domain_reputation_score', 'has_dmarc_record', 'dmarc_enforced', 'spf_pass', 'dkim_pass'
]

features_v2 = real_features.merge(
    domain_table[synthetic_domain_cols], on='sender_domain', how='left'
)

print(features_v2.shape)
print(features_v2.isna().sum())

output_path = f'{BASE_PATH}/features_step2_domain_synthetic.csv'
features_v2.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

(253229, 25)
email_id                       0
source_dataset                 0
label                          0
sender                         0
sender_domain                 43
datetime                       0
day_of_week                    0
hour_of_day_utc                0
is_weekend                     0
sender_email_count             0
sender_active_days             0
sender_avg_daily_volume        0
interarrival_hours         22364
interarrival_ratio         24910
is_burst_anomaly               0
url_count                      0
body_text                      0
domain_age_days               43
tld_phishing_score            43
tld_risk_weight               43
domain_reputation_score       43
has_dmarc_record              43
dmarc_enforced                43
spf_pass                      43
dkim_pass                     43
dtype: int64
Saved to: /content/drive/My Drive/phishing project datasets/processed/features_step2_domain_synthetic.csv


**--------------------------------------------------------------------------------------------------**


In [ ]:
# Real base rates: IPinfo x AbuseIPDB joint RSAC 2026 study (260M+ IPs analyzed)
IP_PARAMS = {
    'phishing': {
        'p_residential_proxy': 0.45,
        'p_vpn_or_anon': 0.53,      # combined VPN/residential-proxy-linked abuse rate
        'p_datacenter_hosting': 0.35 # remaining traffic type, not mutually exclusive w/ above by design
    },
    'benign': {
        # corporate mail (Enron-style) originates from known, static corporate/cloud
        # infrastructure -- residential/VPN traffic would be highly unusual
        'p_residential_proxy': 0.03,
        'p_vpn_or_anon': 0.04,
        'p_datacenter_hosting': 0.85  # corporate mail servers, legitimate cloud providers
    }
}

In [ ]:
def sample_ip_features(label_array, params=IP_PARAMS):
    n = len(label_array)
    is_phish = (label_array == 1)

    p_resid = np.where(is_phish, params['phishing']['p_residential_proxy'], params['benign']['p_residential_proxy'])
    p_vpn = np.where(is_phish, params['phishing']['p_vpn_or_anon'], params['benign']['p_vpn_or_anon'])
    p_dc = np.where(is_phish, params['phishing']['p_datacenter_hosting'], params['benign']['p_datacenter_hosting'])

    ip_is_residential_proxy = np.random.rand(n) < p_resid
    ip_is_vpn_or_anon = np.random.rand(n) < p_vpn
    ip_is_datacenter = np.random.rand(n) < p_dc

    return ip_is_residential_proxy, ip_is_vpn_or_anon, ip_is_datacenter

labels_arr = features_v2['label'].values
resid, vpn, dc = sample_ip_features(labels_arr)

features_v2['ip_is_residential_proxy'] = resid
features_v2['ip_is_vpn_or_anon'] = vpn
features_v2['ip_is_datacenter'] = dc

print(features_v2.groupby('label')[['ip_is_residential_proxy','ip_is_vpn_or_anon','ip_is_datacenter']].mean())

       ip_is_residential_proxy  ip_is_vpn_or_anon  ip_is_datacenter
label                                                              
0                     0.029399           0.039764          0.849854
1                     0.455034           0.536577          0.351342


In [ ]:
def compute_ip_reputation(resid, vpn, dc, noise_std=10):
    n = len(resid)
    base_score = np.full(n, 70.0)  # neutral-ish starting point

    base_score -= resid * 35   # residential-proxy usage is a strong negative signal
    base_score -= vpn * 25     # VPN/anon usage, partially overlapping with resid
    base_score += dc * 10      # legitimate datacenter/hosting origin is a mild positive

    noisy_score = base_score + np.random.normal(0, noise_std, n)
    return np.clip(noisy_score, 0, 100)

features_v2['ip_reputation_score'] = compute_ip_reputation(
    features_v2['ip_is_residential_proxy'].values,
    features_v2['ip_is_vpn_or_anon'].values,
    features_v2['ip_is_datacenter'].values
)

print(features_v2.groupby('label')['ip_reputation_score'].describe())

          count       mean        std  min        25%        50%        75%    max
label                                                                             
0      250249.0  76.401418  12.948029  0.0  69.560908  77.691893  85.141673  100.0
1        2980.0  44.252037  23.967636  0.0  25.936054  44.958182  61.595137  100.0


In [ ]:
output_path = f'{BASE_PATH}/features_step4_ip_synthetic.csv'
features_v2.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(features_v2.shape)

Saved to: /content/drive/My Drive/phishing project datasets/processed/features_step4_ip_synthetic.csv
(253229, 29)


**bold text-------------------------------------------------------------------------------------------------------------------------**

In [ ]:
print(phishtank.shape)
print(phishtank['Label'].value_counts())
print(phishtank[['is_ip_based','subdomain_count','path_length','query_param_count','url_length','uses_https']].describe())

(64280, 11)
Label
1    64280
Name: count, dtype: int64
       subdomain_count   path_length  query_param_count    url_length
count     64280.000000  64280.000000       64280.000000  64280.000000
mean          0.759490     19.564203           0.453905     59.213581
std           0.583533     32.944396           1.363022    156.328256
min           0.000000      0.000000           0.000000     12.000000
25%           0.000000      1.000000           0.000000     25.000000
50%           1.000000      7.000000           0.000000     38.000000
75%           1.000000     18.000000           0.000000     62.000000
max           8.000000    997.000000          44.000000  25523.000000


In [ ]:
# PhishTank is 100% phishing URLs -- it cannot supply benign URL examples.
# Benign URL characteristics below are a documented modeling assumption based on
# general known patterns of legitimate corporate/webmail links (short paths,
# near-universal HTTPS, rarely IP-based, low subdomain depth) -- NOT a cited stat.
BENIGN_URL_PARAMS = {
    'p_uses_https': 0.93,
    'p_is_ip_based': 0.01,
    'subdomain_count_lambda': 1.1,   # Poisson mean
    'path_length_mean': 18, 'path_length_std': 12,
    'query_param_count_lambda': 0.6, # Poisson mean
    'url_length_mean': 45, 'url_length_std': 18,
}

def generate_synthetic_benign_urls(n):
    rng = np.random.default_rng(42)
    return pd.DataFrame({
        'uses_https': rng.random(n) < BENIGN_URL_PARAMS['p_uses_https'],
        'is_ip_based': rng.random(n) < BENIGN_URL_PARAMS['p_is_ip_based'],
        'subdomain_count': rng.poisson(BENIGN_URL_PARAMS['subdomain_count_lambda'], n),
        'path_length': np.clip(rng.normal(BENIGN_URL_PARAMS['path_length_mean'], BENIGN_URL_PARAMS['path_length_std'], n), 0, None).astype(int),
        'query_param_count': rng.poisson(BENIGN_URL_PARAMS['query_param_count_lambda'], n),
        'url_length': np.clip(rng.normal(BENIGN_URL_PARAMS['url_length_mean'], BENIGN_URL_PARAMS['url_length_std'], n), 5, None).astype(int),
    })

In [ ]:
np.random.seed(7)

phishtank_pool = phishtank[['is_ip_based','subdomain_count','path_length','query_param_count','url_length','uses_https']].copy()

has_url_mask = features_v2['url_count'] > 0
phishing_url_mask = has_url_mask & (features_v2['label'] == 1)
benign_url_mask = has_url_mask & (features_v2['label'] == 0)

url_feature_cols = ['is_ip_based','subdomain_count','path_length','query_param_count','url_length','uses_https']

# init all URL feature columns as NaN (represents "no URL present")
for col in url_feature_cols:
    features_v2[col] = np.nan
features_v2['has_url'] = has_url_mask.astype(int)

# phishing emails with URLs: sample real PhishTank rows with replacement
n_phish_urls = phishing_url_mask.sum()
sampled_phish = phishtank_pool.sample(n=n_phish_urls, replace=True, random_state=7).reset_index(drop=True)
features_v2.loc[phishing_url_mask, url_feature_cols] = sampled_phish.values

# benign emails with URLs: draw from the synthetic benign generator
n_benign_urls = benign_url_mask.sum()
synthetic_benign = generate_synthetic_benign_urls(n_benign_urls)
features_v2.loc[benign_url_mask, url_feature_cols] = synthetic_benign[url_feature_cols].values

print(f"Phishing emails w/ URL (real PhishTank rows sampled): {n_phish_urls}")
print(f"Benign emails w/ URL (synthetic rows generated): {n_benign_urls}")
print(f"Emails with no URL (left as NaN/has_url=0): {(~has_url_mask).sum()}")

/tmp/ipykernel_584/1048332039.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, Fa

Phishing emails w/ URL (real PhishTank rows sampled): 2271
Benign emails w/ URL (synthetic rows generated): 36081
Emails with no URL (left as NaN/has_url=0): 214877


In [ ]:
print(features_v2.groupby('label')[url_feature_cols + ['has_url']].mean())

      is_ip_based  subdomain_count  path_length  query_param_count  url_length uses_https   has_url
label                                                                                              
0         0.00995         1.104376    18.014329           0.603254   44.622710   0.930933  0.144180
1        0.004844         0.761339    18.524439           0.410832   56.616028   0.948481  0.762081


In [ ]:
output_path = f'{BASE_PATH}/features_step5_full_synthetic.csv'
features_v2.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(features_v2.shape)
print(features_v2.columns.tolist())

Saved to: /content/drive/My Drive/phishing project datasets/processed/features_step5_full_synthetic.csv
(253229, 36)
['email_id', 'source_dataset', 'label', 'sender', 'sender_domain', 'datetime', 'day_of_week', 'hour_of_day_utc', 'is_weekend', 'sender_email_count', 'sender_active_days', 'sender_avg_daily_volume', 'interarrival_hours', 'interarrival_ratio', 'is_burst_anomaly', 'url_count', 'body_text', 'domain_age_days', 'tld_phishing_score', 'tld_risk_weight', 'domain_reputation_score', 'has_dmarc_record', 'dmarc_enforced', 'spf_pass', 'dkim_pass', 'ip_is_residential_proxy', 'ip_is_vpn_or_anon', 'ip_is_datacenter', 'ip_reputation_score', 'is_ip_based', 'subdomain_count', 'path_length', 'query_param_count', 'url_length', 'uses_https', 'has_url']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from sklearn.model_selection import train_test_split

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'
features_v2 = pd.read_csv(f'{BASE_PATH}/features_step5_full_synthetic.csv')
print(features_v2.shape)
print(features_v2['label'].value_counts(normalize=True))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_707/185844635.py:8: DtypeWarning: Columns (21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  features_v2 = pd.read_csv(f'{BASE_PATH}/features_step5_full_synthetic.csv')


(253229, 36)
label
0    0.988232
1    0.011768
Name: proportion, dtype: float64


In [ ]:
train_df, test_df = train_test_split(
    features_v2,
    test_size=0.20,
    stratify=features_v2['label'],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain label distribution:\n", train_df['label'].value_counts(normalize=True))
print("\nTest label distribution:\n", test_df['label'].value_counts(normalize=True))

Train shape: (202583, 36)
Test shape: (50646, 36)

Train label distribution:
 label
0    0.988232
1    0.011768
Name: proportion, dtype: float64

Test label distribution:
 label
0    0.988232
1    0.011768
Name: proportion, dtype: float64


In [ ]:
train_path = f'{BASE_PATH}/train_features.csv'
test_path = f'{BASE_PATH}/test_features.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved train to: {train_path}")
print(f"Saved test to: {test_path}")

Saved train to: /content/drive/My Drive/phishing project datasets/processed/train_features.csv
Saved test to: /content/drive/My Drive/phishing project datasets/processed/test_features.csv


In [ ]:
import re
from urllib.parse import urlparse

URL_REGEX = re.compile(r'https?://[^\s<>"\')\]]+', re.IGNORECASE)

def extract_first_url_features(body_text):
    if pd.isna(body_text):
        return None
    urls = URL_REGEX.findall(str(body_text))
    if not urls:
        return None
    url = urls[0]
    try:
        parsed = urlparse(url)
        domain = parsed.netloc
        return {
            'is_ip_based': bool(re.match(r'^\d{1,3}(\.\d{1,3}){3}', domain)),
            'subdomain_count': max(domain.count('.') - 1, 0),
            'path_length': len(parsed.path),
            'query_param_count': parsed.query.count('=') if parsed.query else 0,
            'url_length': len(url),
            'uses_https': parsed.scheme == 'https',
        }
    except Exception:
        return None

enron_real_urls = features_v2.loc[
    (features_v2['label'] == 0) & (features_v2['url_count'] > 0), 'body_text'
].apply(extract_first_url_features).dropna()

enron_real_urls_df = pd.DataFrame(list(enron_real_urls))
print(f"Real benign URLs extracted: {len(enron_real_urls_df)}")
print(enron_real_urls_df.describe())

Real benign URLs extracted: 36080
       subdomain_count   path_length  query_param_count    url_length
count     36080.000000  36080.000000       36080.000000  36080.000000
mean          1.174917     14.297062           0.556929     48.482206
std           0.511003     14.156011           1.475495     30.841435
min           0.000000      0.000000           0.000000     10.000000
25%           1.000000      0.000000           0.000000     27.000000
50%           1.000000     12.000000           0.000000     41.000000
75%           1.000000     23.000000           0.000000     60.000000
max           7.000000    136.000000          30.000000    592.000000


In [ ]:
import numpy as np
# time-invariant structural columns: safe to sample directly from real Enron URLs
STRUCTURAL_COLS = ['subdomain_count', 'path_length', 'query_param_count', 'url_length']

# time-dependent columns: 2001 Enron data is not representative of present-day traffic,
# so these are corrected synthetic values instead -- current, real-world legitimate-site norms
BENIGN_PROTOCOL_PARAMS = {
    'p_uses_https': 0.95,   # present-day HTTPS-everywhere norm for legitimate sites
    'p_is_ip_based': 0.005, # legitimate corporate/webmail links are almost never raw-IP
}

def patch_benign_urls_hybrid(df, real_pool, seed):
    df = df.copy()
    mask = (df['label'] == 0) & (df['has_url'] == 1)
    n = mask.sum()
    if n == 0:
        return df

    rng = np.random.default_rng(seed)

    # structural fields: real, resampled from Enron's own extracted URLs
    sampled_structural = real_pool[STRUCTURAL_COLS].sample(n=n, replace=True, random_state=seed).reset_index(drop=True)
    df.loc[mask, STRUCTURAL_COLS] = sampled_structural.values

    # protocol/hosting fields: corrected synthetic, not pulled from 2001 data
    df.loc[mask, 'uses_https'] = rng.random(n) < BENIGN_PROTOCOL_PARAMS['p_uses_https']
    df.loc[mask, 'is_ip_based'] = rng.random(n) < BENIGN_PROTOCOL_PARAMS['p_is_ip_based']

    return df

train_df = patch_benign_urls_hybrid(train_df, enron_real_urls_df, seed=11)
test_df = patch_benign_urls_hybrid(test_df, enron_real_urls_df, seed=22)

print("Train benign URL rows patched:", ((train_df['label']==0) & (train_df['has_url']==1)).sum())
url_feature_cols = ['is_ip_based','subdomain_count','path_length','query_param_count','url_length','uses_https']
print(train_df.groupby('label')[url_feature_cols].mean())

Train benign URL rows patched: 28838
      is_ip_based  subdomain_count  path_length  query_param_count  \
label                                                                
0         0.00586         1.172515    14.372980           0.549865   
1         0.00551         0.763085    18.596694           0.419835   

       url_length uses_https  
label                         
0       48.512865   0.950447  
1       57.380716   0.947107  


In [ ]:
BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'
train_df.to_csv(f'{BASE_PATH}/train_features.csv', index=False)
test_df.to_csv(f'{BASE_PATH}/test_features.csv', index=False)
print("Saved patched train/test splits.")

Saved patched train/test splits.
